# LoRA Contrastive Training for Code-Feedback Alignment

This notebook implements sophisticated contrastive learning using Hugging Face Trainer.

## Architecture
- **Base Model**: Pre-trained code encoder (CodeBERT, StarCoder, etc.)
- **Fine-tuning**: LoRA (Low-Rank Adaptation)
- **Loss**: InfoNCE with in-batch negatives
- **Pooling**: [CLS] token (no mean pooling)

## Negative Sampling Strategies
1. **Random**: Random sampling from batch
2. **Cluster-Based**: Sample from same cluster for hard negatives

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from typing import Dict, List, Optional, Tuple, Union
from dataclasses import dataclass

# Transformers & PEFT
from transformers import (
    AutoModel,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel
)

# Dataset
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Set seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

print("✓ Libraries loaded")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

## 2. Configuration

In [ ]:
@dataclass
class Config:
    # Model
    model_name: str = "microsoft/codebert-base"
    
    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = None
    
    # Training
    output_dir: str = "./checkpoints/lora_contrastive"
    num_train_epochs: int = 10
    per_device_train_batch_size: int = 32
    per_device_eval_batch_size: int = 32
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    logging_steps: int = 50
    eval_steps: int = 200
    save_steps: int = 200
    save_total_limit: int = 3
    
    # InfoNCE
    temperature: float = 0.07
    
    # Data
    max_code_length: int = 512
    max_feedback_length: int = 256
    negative_strategy: str = "cluster"  # "random" or "cluster"
    
    # Paths
    dataset_path: str = "../data/cleaned_dataset_no_cot.csv"
    clustered_dataset_path: str = "../data/dataset_clustered_no_tests.csv"
    
    def __post_init__(self):
        if self.lora_target_modules is None:
            self.lora_target_modules = ["query", "value"]

config = Config()

print("="*100)
print("CONFIGURATION")
print("="*100)
print(f"Model: {config.model_name}")
print(f"LoRA: r={config.lora_r}, alpha={config.lora_alpha}")
print(f"Batch size: {config.per_device_train_batch_size}")
print(f"Learning rate: {config.learning_rate}")
print(f"Temperature: {config.temperature}")
print(f"Negative strategy: {config.negative_strategy}")

## 3. Load Data

In [ ]:
print("="*100)
print("LOADING DATA")
print("="*100)
print()

# Load cleaned dataset
df_clean = pd.read_csv(config.dataset_path)
print(f"✓ Loaded {len(df_clean):,} samples")

# Load clustering info
df_clustered = pd.read_csv(config.clustered_dataset_path)
print(f"✓ Loaded clustering info")

# Merge
df = df_clean.merge(
    df_clustered[['code_id', 'cluster_kmeans']],
    on='code_id',
    how='left'
)
df['cluster_kmeans'] = df['cluster_kmeans'].fillna(-1).astype(int)

print(f"✓ Merged dataset: {len(df):,} samples")
print(f"\nCluster distribution:")
print(df['cluster_kmeans'].value_counts().sort_index().head(15))

# Split dataset
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['cluster_kmeans'])
train_df, val_df = train_test_split(train_df, test_size=0.111, random_state=42, stratify=train_df['cluster_kmeans'])

print(f"\nSplits:")
print(f"  Train: {len(train_df):,}")
print(f"  Val:   {len(val_df):,}")
print(f"  Test:  {len(test_df):,}")

## 4. Custom Dataset

In [ ]:
class ContrastiveDataset(Dataset):
    """Dataset for contrastive learning."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        
        # Build cluster index
        self.cluster_to_indices = {}
        for idx, cluster in enumerate(df['cluster_kmeans']):
            if cluster not in self.cluster_to_indices:
                self.cluster_to_indices[cluster] = []
            self.cluster_to_indices[cluster].append(idx)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'code': row['code_snippet'],
            'feedback': row['generated_feedback'],
            'cluster': row['cluster_kmeans'],
            'idx': idx
        }
    
    def get_cluster_batch_indices(self, batch_size: int) -> List[int]:
        """Get batch indices from same cluster."""
        cluster = np.random.choice(list(self.cluster_to_indices.keys()))
        cluster_indices = self.cluster_to_indices[cluster]
        
        if len(cluster_indices) < batch_size:
            sampled = cluster_indices.copy()
            remaining = batch_size - len(sampled)
            all_indices = list(range(len(self.df)))
            random_indices = np.random.choice(
                [i for i in all_indices if i not in sampled],
                size=remaining,
                replace=False
            )
            sampled.extend(random_indices.tolist())
        else:
            sampled = np.random.choice(cluster_indices, size=batch_size, replace=False).tolist()
        
        return sampled

# Create datasets
train_dataset = ContrastiveDataset(train_df)
val_dataset = ContrastiveDataset(val_df)
test_dataset = ContrastiveDataset(test_df)

print(f"✓ Datasets created")
print(f"  Train: {len(train_dataset):,}")
print(f"  Val: {len(val_dataset):,}")
print(f"  Test: {len(test_dataset):,}")

## 5. Data Collator

In [ ]:
class ContrastiveCollator:
    """Collator for contrastive learning."""
    
    def __init__(
        self,
        tokenizer,
        max_code_length: int = 512,
        max_feedback_length: int = 256
    ):
        self.tokenizer = tokenizer
        self.max_code_length = max_code_length
        self.max_feedback_length = max_feedback_length
    
    def __call__(self, batch):
        codes = [item['code'] for item in batch]
        feedbacks = [item['feedback'] for item in batch]
        clusters = torch.tensor([item['cluster'] for item in batch])
        indices = torch.tensor([item['idx'] for item in batch])
        
        # Tokenize codes
        code_encodings = self.tokenizer(
            codes,
            padding=True,
            truncation=True,
            max_length=self.max_code_length,
            return_tensors='pt'
        )
        
        # Tokenize feedbacks
        feedback_encodings = self.tokenizer(
            feedbacks,
            padding=True,
            truncation=True,
            max_length=self.max_feedback_length,
            return_tensors='pt'
        )
        
        return {
            'code_input_ids': code_encodings['input_ids'],
            'code_attention_mask': code_encodings['attention_mask'],
            'feedback_input_ids': feedback_encodings['input_ids'],
            'feedback_attention_mask': feedback_encodings['attention_mask'],
            'clusters': clusters,
            'indices': indices
        }

print("✓ Collator defined")

## 6. Contrastive Model

In [ ]:
class ContrastiveModel(nn.Module):
    """Dual encoder with LoRA for contrastive learning."""
    
    def __init__(self, base_model_name: str, lora_config: LoraConfig):
        super().__init__()
        
        # Load base model
        self.encoder = AutoModel.from_pretrained(base_model_name)
        
        # Apply LoRA
        self.encoder = get_peft_model(self.encoder, lora_config)
        
        print("\nLoRA configuration:")
        self.encoder.print_trainable_parameters()
    
    def encode(self, input_ids, attention_mask):
        """Encode input using [CLS] token."""
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use [CLS] token (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (batch_size, hidden_dim)
        
        # L2 normalize
        cls_embedding = F.normalize(cls_embedding, p=2, dim=1)
        
        return cls_embedding
    
    def forward(
        self,
        code_input_ids=None,
        code_attention_mask=None,
        feedback_input_ids=None,
        feedback_attention_mask=None,
        **kwargs
    ):
        # Encode code and feedback
        code_embeddings = self.encode(code_input_ids, code_attention_mask)
        feedback_embeddings = self.encode(feedback_input_ids, feedback_attention_mask)
        
        return {
            'code_embeddings': code_embeddings,
            'feedback_embeddings': feedback_embeddings
        }

print("✓ Model class defined")

## 7. Custom Trainer with InfoNCE Loss

In [ ]:
class ContrastiveTrainer(Trainer):
    """Custom Trainer with InfoNCE loss."""
    
    def __init__(self, *args, temperature=0.07, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Compute InfoNCE loss with in-batch negatives.
        """
        # Forward pass
        outputs = model(**inputs)
        code_embeddings = outputs['code_embeddings']
        feedback_embeddings = outputs['feedback_embeddings']
        
        batch_size = code_embeddings.shape[0]
        
        # Compute similarity matrix: (batch_size, batch_size)
        similarity = torch.matmul(code_embeddings, feedback_embeddings.T) / self.temperature
        
        # Labels: diagonal is positive
        labels = torch.arange(batch_size, device=similarity.device)
        
        # InfoNCE loss (bi-directional)
        loss_c2f = F.cross_entropy(similarity, labels)
        loss_f2c = F.cross_entropy(similarity.T, labels)
        loss = (loss_c2f + loss_f2c) / 2
        
        # Compute metrics for logging
        with torch.no_grad():
            # Positive similarities
            positive_sim = torch.diagonal(similarity).mean()
            
            # Negative similarities
            mask = torch.eye(batch_size, device=similarity.device).bool()
            negative_sim = similarity.masked_select(~mask).mean()
            
            # Margin
            margin = positive_sim - negative_sim
            
            # Accuracy
            preds = similarity.argmax(dim=1)
            accuracy = (preds == labels).float().mean()
            
            # Store metrics for logging
            self.log({
                'positive_sim': positive_sim.item(),
                'negative_sim': negative_sim.item(),
                'margin': margin.item(),
                'accuracy': accuracy.item()
            })
        
        return (loss, outputs) if return_outputs else loss

print("✓ Custom Trainer defined")

## 8. Compute Metrics

In [ ]:
def compute_metrics(eval_pred: EvalPrediction) -> Dict[str, float]:
    """
    Compute custom metrics for evaluation.
    
    Note: Since we're doing contrastive learning, metrics are computed
    in the compute_loss method. This function is kept for compatibility.
    """
    # Metrics are logged in compute_loss
    return {}

print("✓ Metrics function defined")

## 9. Initialize Model and Tokenizer

In [ ]:
print("="*100)
print("INITIALIZING MODEL")
print("="*100)
print()

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✓ Tokenizer: {config.model_name}")

# LoRA config
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=config.lora_target_modules,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)
print(f"✓ LoRA config: r={config.lora_r}, alpha={config.lora_alpha}")

# Model
model = ContrastiveModel(
    base_model_name=config.model_name,
    lora_config=lora_config
)
print(f"✓ Model initialized")

## 10. Data Collator Instance

In [ ]:
# Create collator
data_collator = ContrastiveCollator(
    tokenizer=tokenizer,
    max_code_length=config.max_code_length,
    max_feedback_length=config.max_feedback_length
)

print("✓ Data collator created")

## 11. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=config.warmup_steps,
    logging_dir=f"{config.output_dir}/logs",
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to=["tensorboard"],
    remove_unused_columns=False,  # Important for custom inputs
)

print("="*100)
print("TRAINING ARGUMENTS")
print("="*100)
print(f"Output dir: {training_args.output_dir}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Warmup steps: {training_args.warmup_steps}")
print(f"FP16: {training_args.fp16}")

## 12. Initialize Trainer

In [ ]:
trainer = ContrastiveTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    temperature=config.temperature
)

print("="*100)
print("TRAINER INITIALIZED")
print("="*100)
print(f"Temperature: {config.temperature}")
print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples: {len(val_dataset):,}")
print(f"\nReady to train!")

## 13. Train Model

In [ ]:
print("="*100)
print("STARTING TRAINING")
print("="*100)
print()

# Train
train_result = trainer.train()

print("\n" + "="*100)
print("TRAINING COMPLETE")
print("="*100)
print(f"\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

## 14. Save Model

In [ ]:
# Save the best model
best_model_path = Path(config.output_dir) / "best_model"
trainer.save_model(best_model_path)

print(f"✓ Best model saved to: {best_model_path}")

# Save training metrics
import json
metrics_path = Path(config.output_dir) / "training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(train_result.metrics, f, indent=2)

print(f"✓ Metrics saved to: {metrics_path}")

## 15. Evaluate on Test Set

In [ ]:
print("="*100)
print("EVALUATING ON TEST SET")
print("="*100)
print()

# Evaluate
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\nTest results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

# Save test results
test_results_path = Path(config.output_dir) / "test_results.json"
with open(test_results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n✓ Test results saved to: {test_results_path}")

## 16. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator

# Read TensorBoard logs
log_dir = Path(config.output_dir) / "logs"

def read_tensorboard_logs(log_dir):
    """Read metrics from TensorBoard logs."""
    ea = event_accumulator.EventAccumulator(str(log_dir))
    ea.Reload()
    
    metrics = {}
    for tag in ea.Tags()['scalars']:
        events = ea.Scalars(tag)
        metrics[tag] = [(e.step, e.value) for e in events]
    
    return metrics

try:
    # Find the latest run
    run_dirs = list(log_dir.glob('*'))
    if run_dirs:
        latest_run = max(run_dirs, key=lambda p: p.stat().st_mtime)
        metrics = read_tensorboard_logs(latest_run)
        
        # Plot
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Loss
        if 'loss' in metrics:
            steps, values = zip(*metrics['loss'])
            axes[0, 0].plot(steps, values)
            axes[0, 0].set_xlabel('Step')
            axes[0, 0].set_ylabel('Loss')
            axes[0, 0].set_title('Training Loss')
            axes[0, 0].grid(True, alpha=0.3)
        
        # Margin
        if 'margin' in metrics:
            steps, values = zip(*metrics['margin'])
            axes[0, 1].plot(steps, values, color='green')
            axes[0, 1].set_xlabel('Step')
            axes[0, 1].set_ylabel('Margin')
            axes[0, 1].set_title('Positive-Negative Margin')
            axes[0, 1].grid(True, alpha=0.3)
        
        # Accuracy
        if 'accuracy' in metrics:
            steps, values = zip(*metrics['accuracy'])
            axes[1, 0].plot(steps, values, color='red')
            axes[1, 0].set_xlabel('Step')
            axes[1, 0].set_ylabel('Accuracy')
            axes[1, 0].set_title('Retrieval Accuracy')
            axes[1, 0].grid(True, alpha=0.3)
        
        # Similarities
        if 'positive_sim' in metrics and 'negative_sim' in metrics:
            steps_pos, values_pos = zip(*metrics['positive_sim'])
            steps_neg, values_neg = zip(*metrics['negative_sim'])
            axes[1, 1].plot(steps_pos, values_pos, label='Positive', color='green')
            axes[1, 1].plot(steps_neg, values_neg, label='Negative', color='orange')
            axes[1, 1].set_xlabel('Step')
            axes[1, 1].set_ylabel('Similarity')
            axes[1, 1].set_title('Positive vs Negative Similarity')
            axes[1, 1].legend()
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plot_path = Path(config.output_dir) / 'training_curves.png'
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"✓ Training curves saved to: {plot_path}")
    else:
        print("No TensorBoard logs found")
        
except Exception as e:
    print(f"Could not load TensorBoard logs: {e}")
    print("You can visualize training with: tensorboard --logdir", log_dir)

## 17. Inference Example

In [ ]:
print("="*100)
print("INFERENCE EXAMPLE")
print("="*100)
print()

# Put model in eval mode
model.eval()

# Example code and feedback
example_code = """int factorial(int n) {
    if (n == 0) return 1;
    return n * factorial(n - 1);
}"""

example_feedback = "Consider the behavior when n is negative."

# Tokenize
code_inputs = tokenizer(
    example_code,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=config.max_code_length
)

feedback_inputs = tokenizer(
    example_feedback,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=config.max_feedback_length
)

# Move to device
device = next(model.parameters()).device
code_inputs = {k: v.to(device) for k, v in code_inputs.items()}
feedback_inputs = {k: v.to(device) for k, v in feedback_inputs.items()}

# Encode
with torch.no_grad():
    code_embedding = model.encode(code_inputs['input_ids'], code_inputs['attention_mask'])
    feedback_embedding = model.encode(feedback_inputs['input_ids'], feedback_inputs['attention_mask'])
    
    # Compute similarity
    similarity = torch.matmul(code_embedding, feedback_embedding.T)

print(f"Code:\n{example_code}")
print(f"\nFeedback:\n{example_feedback}")
print(f"\nSimilarity score: {similarity.item():.4f}")
print(f"\nCode embedding shape: {code_embedding.shape}")
print(f"Feedback embedding shape: {feedback_embedding.shape}")

## Summary

This notebook implements:

1. ✓ **LoRA fine-tuning** with PEFT
2. ✓ **InfoNCE loss** with in-batch negatives
3. ✓ **Custom Trainer** for contrastive learning
4. ✓ **[CLS] token encoding** (no mean pooling)
5. ✓ **Custom metrics**: margin, accuracy, similarities
6. ✓ **Cluster-based sampling** for hard negatives
7. ✓ **TensorBoard logging** for visualization

**Next steps**:
- Implement cluster-based batch sampler
- Compare random vs cluster sampling strategies
- Add retrieval metrics (MRR, NDCG)
- Experiment with different temperatures
- Try different base models